In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_PATH = "/kaggle/input/competitions/feedback-prize-english-language-learning"
MODEL_PATH = "/kaggle/input/datasets/jonathanchan/deberta-v3-base/deberta-v3-base"
TARGETS = ["cohesion", "syntax", "vocabulary", "phraseology", "grammar", "conventions"]
MAX_LEN = 512
BATCH_SIZE = 8
EMBED_DIM = 768

In [2]:
train_df = pd.read_csv(f"{DATA_PATH}/train.csv")
test_df  = pd.read_csv(f"{DATA_PATH}/test.csv")

print(train_df.shape, test_df.shape)
print(train_df.head(2))

(3911, 8) (3, 2)
        text_id                                          full_text  cohesion  \
0  0016926B079C  I think that students would benefit from learn...       3.5   
1  0022683E9EA5  When a problem is a change you have to let it ...       2.5   

   syntax  vocabulary  phraseology  grammar  conventions  
0     3.5         3.0          3.0      4.0          3.0  
1     2.5         3.0          2.0      2.0          2.5  


In [4]:
class EmbeddingModel(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_path)

    def mean_pooling(self, last_hidden, attention_mask):
        mask_expanded = attention_mask.unsqueeze(-1).float()
        return (last_hidden * mask_expanded).sum(1) / mask_expanded.sum(1).clamp(min=1e-9)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        return self.mean_pooling(out.last_hidden_state, attention_mask)


class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return enc["input_ids"].squeeze(), enc["attention_mask"].squeeze()


def get_embeddings(texts, model, tokenizer, batch_size=8):
    dataset = TextDataset(texts, tokenizer, MAX_LEN)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    model.eval()
    all_embeds = []
    with torch.no_grad():
        for input_ids, attention_mask in loader:
            input_ids      = input_ids.to(DEVICE)
            attention_mask = attention_mask.to(DEVICE)
            embeds = model(input_ids, attention_mask)
            all_embeds.append(embeds.cpu().numpy())
    return np.vstack(all_embeds)

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
embed_model = EmbeddingModel(MODEL_PATH).to(DEVICE)

print("Encoding train texts...")
train_embeds = get_embeddings(train_df["full_text"].tolist(), embed_model, tokenizer, BATCH_SIZE)

print("Encoding test texts...")
test_embeds  = get_embeddings(test_df["full_text"].tolist(),  embed_model, tokenizer, BATCH_SIZE)

print("Train embeddings shape:", train_embeds.shape)
print("Test  embeddings shape:", test_embeds.shape)

The tokenizer you are loading from '/kaggle/input/datasets/jonathanchan/deberta-v3-base/deberta-v3-base' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: /kaggle/input/datasets/jonathanchan/deberta-v3-base/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifer.bias         | UNEXPECTED |  | 
mask_predictions.classifer.weight       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding train texts...
Encoding test texts...
Train embeddings shape: (3911, 768)
Test  embeddings shape: (3, 768)


In [6]:
from sklearn.neighbors import NearestNeighbors

# Normalize for cosine similarity
def normalize(embeds):
    norms = np.linalg.norm(embeds, axis=1, keepdims=True)
    return embeds / (norms + 1e-9)

train_embeds_norm = normalize(train_embeds).astype("float32")
test_embeds_norm  = normalize(test_embeds).astype("float32")

# Build KNN index using sklearn instead of faiss
nbrs = NearestNeighbors(n_neighbors=20, metric="cosine", algorithm="brute", n_jobs=-1)
nbrs.fit(train_embeds_norm)
print(f"KNN index built with {len(train_embeds_norm)} vectors")

KNN index built with 3911 vectors


In [7]:
def rag_predict(test_embeds, nbrs, train_labels, k=20):
    distances, indices = nbrs.kneighbors(test_embeds)
    similarities = 1 - distances  # cosine distance -> similarity
    preds = []
    for i in range(len(test_embeds)):
        weights = np.exp(similarities[i] * 10)
        weights = weights / weights.sum()
        neighbor_labels = train_labels[indices[i]]
        pred = (neighbor_labels * weights[:, None]).sum(axis=0)
        preds.append(pred)
    return np.array(preds)

train_labels = train_df[TARGETS].values
test_preds = rag_predict(test_embeds_norm, nbrs, train_labels, k=20)
print("Predictions shape:", test_preds.shape)

Predictions shape: (3, 6)


In [8]:
VAL_SIZE = 500
val_embeds_norm  = train_embeds_norm[-VAL_SIZE:]
val_labels       = train_labels[-VAL_SIZE:]

val_nbrs = NearestNeighbors(n_neighbors=20, metric="cosine", algorithm="brute", n_jobs=-1)
val_nbrs.fit(train_embeds_norm[:-VAL_SIZE])
val_preds = rag_predict(val_embeds_norm, val_nbrs, train_labels[:-VAL_SIZE], k=20)

mcrmse = np.mean([
    np.sqrt(mean_squared_error(val_labels[:, i], val_preds[:, i]))
    for i in range(len(TARGETS))
])
print(f"Validation MCRMSE: {mcrmse:.4f}")

Validation MCRMSE: 0.5445


In [9]:
submission = pd.DataFrame(test_preds, columns=TARGETS)
submission.insert(0, "text_id", test_df["text_id"].values)

# Clip predictions to valid score range [1.0, 5.0]
for col in TARGETS:
    submission[col] = submission[col].clip(1.0, 5.0)

submission.to_csv("submission.csv", index=False)
print("submission.csv saved!")
print(submission.head())

submission.csv saved!
        text_id  cohesion    syntax  vocabulary  phraseology   grammar  \
0  0000C359D63E  2.798462  2.598484    3.123698     2.850126  2.747591   
1  000BAD50D026  2.997585  2.768001    2.920220     2.717387  2.596521   
2  00367BB2546B  3.300443  3.099804    3.375526     3.301716  3.099703   

   conventions  
0     2.722837  
1     2.873513  
2     3.125752  
